# Section 3 - Model Creation, Hyperparameter Search, and Model Evaluation 

**This is an attempt to create a model using GradientBoostingClassifier but did not get to finish given the time constraints.**

1. Train/test split
2. Implement RandomForestClassifier 
3. Hyperparameter Tuning using RandomizedSearchCV
4. Retrain model using final hyperparameters
5. Generate new predictions for this new model
4. F1 score evaluation

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# accuracy metrics
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# train/test/CV split and tuning tools
from sklearn.model_selection import train_test_split, validation_curve
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import GradientBoostingClassifier

# used to measure how long each search takes
import time

In [7]:
import os
print(os.getcwd())

/Users/sa11/TKH/Projects/Fraud_Detection_Project/code/other_model


In [8]:
df = pd.read_csv('../../data/clean_bank_transactions.csv')
df.head()

,step,amount,isFraud,balance_change_orig,balance_change_dest,balance_mismatch,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,0,-9839.64,0.0,1.455192e-11,0,0,1,0
1,1,1864.28,0,-1864.28,0.0,1.136868e-12,0,0,1,0
2,1,181.00,1,-181.00,0.0,0.000000e+00,0,0,0,1
3,1,181.00,1,-181.00,-21182.0,0.000000e+00,1,0,0,0
4,1,11668.14,0,-11668.14,0.0,0.000000e+00,0,0,1,0


In [3]:
df.shape

(6362620, 10)

In [9]:
X = df.drop(columns=['isFraud'])
y = df['isFraud']

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Train: ", X_train.shape)
print("Test: ", X_test.shape)
print("Fraud cases in training set: ", y_train.sum())
print("Fraud cases in test set: ", y_test.sum())

Train:  (5090096, 9)
Test:  (1272524, 9)
Fraud cases in training set:  6570
Fraud cases in test set:  1643


### Baseline - Gradient Boost Model

In [6]:
start = time.time()
gb_baseline = GradientBoostingClassifier(random_state=42)
gb_baseline.fit(X_train, y_train)

elapsed_grid = time.time() - start

yhat_baseline = gb_baseline.predict(X_test)

confusion = confusion_matrix(y_test, yhat_baseline)

f1_score_baseline = f1_score(y_test, yhat_baseline)

print(f"Time:     {elapsed_grid:.2f}s")
print(f"F1 Score: {f1_score_baseline:.4f}")
print()

print("Confusion Matrix \n", confusion)
print()

print(f"\nClassification Report:\n{classification_report(y_test, yhat_baseline)}")

#F1 Score: 0.3682

Time:     2439.49s
F1 Score: 0.3682

Confusion Matrix 
 [[1270800      81]
 [   1254     389]]


Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.83      0.24      0.37      1643

    accuracy                           1.00   1272524
   macro avg       0.91      0.62      0.68   1272524
weighted avg       1.00      1.00      1.00   1272524



In [7]:
param_dist = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'subsample': [0.6, 0.8, 1.0],
}

random_search = RandomizedSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=5,               # number of random combinations to try
    scoring='f1',            # optimize for F1 since data is imbalanced
    cv=5,                    # 5-fold cross validation
    verbose=2,
    random_state=42,
    n_jobs=-1                # use all CPU cores
)

random_search.fit(X_train, y_train)
print("Best params:", random_search.best_params_)

Fitting 5 folds for each of 5 candidates, totalling 25 fits
[CV] END learning_rate=0.2, max_depth=4, min_samples_leaf=4, min_samples_split=10, n_estimators=100, subsample=0.8; total time=44.6min
[CV] END learning_rate=0.2, max_depth=4, min_samples_leaf=4, min_samples_split=10, n_estimators=100, subsample=0.8; total time=44.7min
[CV] END learning_rate=0.2, max_depth=4, min_samples_leaf=4, min_samples_split=10, n_estimators=100, subsample=0.8; total time=44.7min
[CV] END learning_rate=0.2, max_depth=4, min_samples_leaf=4, min_samples_split=10, n_estimators=100, subsample=0.8; total time=44.8min
[CV] END learning_rate=0.2, max_depth=4, min_samples_leaf=4, min_samples_split=10, n_estimators=100, subsample=0.8; total time=44.8min
[CV] END learning_rate=0.1, max_depth=5, min_samples_leaf=2, min_samples_split=10, n_estimators=200, subsample=1.0; total time=142.1min
[CV] END learning_rate=0.1, max_depth=5, min_samples_leaf=2, min_samples_split=10, n_estimators=200, subsample=1.0; total time=14

KeyboardInterrupt: 